# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [5]:
# 1. 기존의 꼬인 패키지들을 삭제
!pip uninstall -y transformers accelerate peft bitsandbytes

# 2. 5.0.0 미만(<5.0.0) 중 가장 최신 안정 버전을 설치
# 핵심: "transformers<5.0.0"으로 범위를 지정합니다.
!pip install -q "transformers<5.0.0" accelerate peft bitsandbytes datasets

# 3. 코랩 호환을 위한 pandas 및 pillow 버전 조정
!pip install -q "pandas==2.2.2" "pillow<11.0.0"

Found existing installation: transformers 5.4.0
Uninstalling transformers-5.4.0:
  Successfully uninstalled transformers-5.4.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 52.8 MB/s eta 0:00:00


In [49]:
!pip install unsloth
# 최신 비전 모델 지원을 위한 추가 설치
!pip install unsloth-vision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 133.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 28.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Succ

ERROR: Could not find a version that satisfies the requirement unsloth-vision (from versions: none)
ERROR: No matching distribution found for unsloth-vision


In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [23]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 압축 해제
!unzip "/content/260401_15_2_ai_데이터배포용.zip" -d "/content/"

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/train/train_0074.jpg  
  inflating: /content/train/train_0075.jpg  
  inflating: /content/train/train_0076.jpg  
  inflating: /content/train/train_0077.jpg  
  inflating: /content/train/train_0078.jpg  
  inflating: /content/train/train_0079.jpg  
  inflating: /content/train/train_0080.jpg  
  inflating: /content/train/train_0081.jpg  
  inflating: /content/train/train_0082.jpg  
  inflating: /content/train/train_0083.jpg  
  inflating: /content/train/train_0084.jpg  
  inflating: /content/train/train_0085.jpg  
  inflating: /content/train/train_0086.jpg  
  inflating: /content/train/train_0087.jpg  
  inflating: /content/train/train_0088.jpg  
  inflating: /content/train/train_0089.jpg  
  inflating: /content/train/train_0090.jpg  
  inflating: /content/train/train_0091.jpg  
  inflating: /content/train/train_0092.jpg  
  inflating: /content/train/train_0093.jpg  
  inflating: /content/train/train_0094.jpg  
  inflating: /conte

# 라이브러리, 데이터, 설정

In [25]:
import transformers
print(transformers.__version__)

4.57.6


In [9]:
# ============================================================
# Block 2: Import 및 설정
# ============================================================

import os, re, math, random, json
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any

import torch
from torch.utils.data import Dataset, DataLoader

# [수정] AutoModelForVision2Seq → Qwen2_5_VLForConditionalGeneration
# AutoModelForVision2Seq는 v5.0에서 제거 예정 (FutureWarning 발생)
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device    : {device}")
print(f"PyTorch   : {torch.__version__}")

# ── 경로 설정 ────────────────────────────────────────────────
BASE_DIR   = Path("/content")
TRAIN_CSV  = BASE_DIR / "train.csv"
TEST_CSV   = BASE_DIR / "test.csv"
CKPT_DIR   = BASE_DIR / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

# ── 하이퍼파라미터 ───────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-VL-7B-Instruct"
IMAGE_SIZE     = 384
MAX_NEW_TOKENS = 2

BATCH_SIZE     = 6
GRAD_ACCUM     = 1
EPOCHS         = 3
LR             = 2e-4
WEIGHT_DECAY   = 0.01
WARMUP_RATIO   = 0.1
MAX_GRAD_NORM  = 1.0
PATIENCE       = 2

# [수정] LoRA r=16~24 범위로 조정 (피드백: 효율/일반화 균형)
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05

TRAIN_RATIO    = 0.9
BEST_SAVE_DIR  = str(CKPT_DIR / "best_model")

# ── 실험 config 저장 (피드백: 운영/롤백성) ───────────────────
CONFIG = {
    "model_id"     : MODEL_ID,
    "image_size"   : IMAGE_SIZE,
    "batch_size"   : BATCH_SIZE,
    "grad_accum"   : GRAD_ACCUM,
    "epochs"       : EPOCHS,
    "lr"           : LR,
    "warmup_ratio" : WARMUP_RATIO,
    "lora_r"       : LORA_R,
    "lora_alpha"   : LORA_ALPHA,
    "timestamp"    : datetime.now().strftime("%Y%m%d_%H%M%S"),
}
config_path = CKPT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)
print(f"Config saved → {config_path}")
print(json.dumps(CONFIG, indent=2))

# ── 함수 정의: 데이터 전처리 및 파이프라인 (Member 1) ──────────────────────────
from collections import Counter
import os
from tqdm.auto import tqdm
from PIL import Image

def filter_high_quality_dev(dev_df, agreement_threshold=4):
    final_answers = []
    keep_indices = []
    for idx, row in dev_df.iterrows():
        answers = [str(val).strip().lower() for val in row[['answer1', 'answer2', 'answer3', 'answer4', 'answer5']] if pd.notna(val)]
        if not answers: continue
        counter = Counter(answers)
        top_ans, top_count = counter.most_common(1)[0]
        if top_count >= agreement_threshold:
            final_answers.append(top_ans)
            keep_indices.append(idx)
    filtered_df = dev_df.loc[keep_indices].copy()
    filtered_df['answer'] = final_answers
    print(f"원본 Dev 데이터: {len(dev_df)}개 -> 고품질 Dev 데이터: {len(filtered_df)}개")
    return filtered_df.reset_index(drop=True)

def precompute_images(df, img_col, dest_dir, img_size=(384, 384)):
    os.makedirs(dest_dir, exist_ok=True)
    cached_paths = []
    print(f"[{dest_dir}] 이미지 캐싱 중 (Epoch 반복 연산 제거)...")
    for path in tqdm(df[img_col]):
        # BASE_DIR이 적용된 절대 경로로 읽어옵니다.
        full_path = BASE_DIR / path
        filename = os.path.basename(path)
        dest_path = os.path.join(dest_dir, filename)

        if not os.path.exists(dest_path):
            img = Image.open(full_path).convert('RGB')
            # Block 3에 정의되어 있는 letterbox_image 함수를 사용합니다.
            img = letterbox_image(img, target_size=img_size[0])
            img.save(dest_path)
        cached_paths.append(dest_path)
    return cached_paths

# ── 데이터 로드 + Pre-computation 파이프라인 가동 ──────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
dev_df   = pd.read_csv(BASE_DIR / "dev.csv") # dev.csv 로드 추가
test_df  = pd.read_csv(TEST_CSV)

# 1. Dev 노이즈 필터링 (다수결 4명 이상 일치하는 고품질 데이터만 검증셋으로 활용)
valid_subset = filter_high_quality_dev(dev_df, agreement_threshold=4)

# train_test_split을 하지 않고, train_df 전체를 학습에 쏟아붓습니다.
train_subset = train_df

# 2. 이미지 패딩 1회성 미리 완료하기 (데이터 파이프라인 병목 제거)
train_subset['cached_path'] = precompute_images(train_subset, 'path', BASE_DIR / 'train_cached', (IMAGE_SIZE, IMAGE_SIZE))
valid_subset['cached_path'] = precompute_images(valid_subset, 'path', BASE_DIR / 'dev_cached', (IMAGE_SIZE, IMAGE_SIZE))
test_df['cached_path']      = precompute_images(test_df, 'path', BASE_DIR / 'test_cached', (IMAGE_SIZE, IMAGE_SIZE))

print(f"\n최종 파이프라인 세팅 완료 - Train: {len(train_subset)} | Valid(Dev): {len(valid_subset)}")
print(f"Train 분포:\n{train_subset['answer'].value_counts().sort_index()}")
print(f"Valid 분포:\n{valid_subset['answer'].value_counts().sort_index()}")


Device    : cuda
PyTorch   : 2.10.0+cu128
Config saved → /content/checkpoints/config.json
{
  "model_id": "Qwen/Qwen2.5-VL-7B-Instruct",
  "image_size": 384,
  "batch_size": 6,
  "grad_accum": 1,
  "epochs": 3,
  "lr": 0.0002,
  "warmup_ratio": 0.1,
  "lora_r": 16,
  "lora_alpha": 32,
  "timestamp": "20260402_061732"
}
원본 Dev 데이터: 4413개 -> 고품질 Dev 데이터: 902개
[/content/train_cached] 이미지 캐싱 중 (Epoch 반복 연산 제거)...


  0%|          | 0/5073 [00:00<?, ?it/s]

[/content/dev_cached] 이미지 캐싱 중 (Epoch 반복 연산 제거)...


  0%|          | 0/902 [00:00<?, ?it/s]

[/content/test_cached] 이미지 캐싱 중 (Epoch 반복 연산 제거)...


  0%|          | 0/5074 [00:00<?, ?it/s]


최종 파이프라인 세팅 완료 - Train: 5073 | Valid(Dev): 902
Train 분포:
answer
a    1263
b    1265
c    1311
d    1234
Name: count, dtype: int64
Valid 분포:
answer
a    184
b    188
c    117
d    413
Name: count, dtype: int64


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [2]:
# ============================================================
# Block 3: 모델 & Processor 로드 (Unsloth + Flash Attention 2 적용)
# ============================================================
from unsloth import FastVisionModel
import torch

# 1. Unsloth를 통한 모델 및 Processor 로드 (QLoRA 4bit + Flash Attention 2 자동 적용)
# 기존 BitsAndBytesConfig와 prepare_model_for_kbit_training을 대체합니다.
model, processor = FastVisionModel.from_pretrained(
    model_name                 = MODEL_ID,
    load_in_4bit               = True,      # 4bit 양자화 활성화
    use_gradient_checkpointing = "unsloth", # Unsloth 특화 메모리 절약 체크포인팅 (필수)
)

# (선택 유지) 기존처럼 Processor에 이미지 사이즈 제한을 강제하고 싶을 경우
processor.image_processor.min_pixels = IMAGE_SIZE * IMAGE_SIZE
processor.image_processor.max_pixels = IMAGE_SIZE * IMAGE_SIZE

# Vision Encoder 동결 (기존 코드 유지: 학습 효율을 위해 비전 파트는 동결)
frozen_cnt = 0
for name, param in model.visual.named_parameters():
    param.requires_grad = False
    frozen_cnt += 1
print(f"❄️  Frozen {frozen_cnt} tensors in visual encoder")

# 2. Unsloth 최적화 LoRA 어댑터 적용
# 일반 peft.get_peft_model 대신 FastVisionModel.get_peft_model을 사용합니다.
model = FastVisionModel.get_peft_model(
    model,
    r               = LORA_R,
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = 0,          # [핵심 수정] Unsloth 최고 속도 최적화를 위해 dropout은 0으로 설정해야 합니다.
    bias            = "none",
    use_rslora      = True,       # 기존 설정 유지 (rslora 적용)
    target_modules  = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model.print_trainable_parameters()

/tmp/ipykernel_72761/2656464214.py:4: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastVisionModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen2_5_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.90G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

❄️  Frozen 390 tensors in visual encoder
trainable params: 47,589,376 || all params: 8,339,756,032 || trainable%: 0.5706


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [32]:
# ============================================================
# Block 4: 프롬프트 템플릿 (유지)
# ============================================================

SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )


In [3]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

PROMPT_TEMPLATES = [
    # Template 0 (기존)
    "{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.",
    # Template 1 (변형 A)
    "다음 질문을 읽고 이미지에 가장 적절한 선택지를 고르세요.\n질문: {question}\n\n[선택지]\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n단 하나의 알파벳(a, b, c, d)만 대답하세요.",
    # Template 2 (변형 B)
    "Question: {question}\nOptions:\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n위 옵션 중 정답에 해당하는 기호 하나(a/b/c/d)를 출력하시오."
]

def build_mc_prompt(question, a, b, c, d, template_idx=0):
    """지정된 템플릿 인덱스를 사용하여 프롬프트를 생성합니다."""
    template = PROMPT_TEMPLATES[template_idx % len(PROMPT_TEMPLATES)]
    return template.format(question=question, a=a, b=b, c=c, d=d)

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [4]:
# ============================================================
# Block 5: letterbox + Dataset + Collator
# ============================================================

def letterbox_image(img: Image.Image, target_size: int = 384) -> Image.Image:
    """원본 비율 유지 + 검정 패딩 (유지 - train/valid/infer 동일 파이프라인)"""
    w, h   = img.size
    scale  = target_size / max(w, h)
    new_w  = int(w * scale)
    new_h  = int(h * scale)
    resized = img.resize((new_w, new_h), Image.LANCZOS)
    canvas  = Image.new("RGB", (target_size, target_size), (0, 0, 0))
    canvas.paste(resized, ((target_size - new_w) // 2, (target_size - new_h) // 2))
    return canvas


# ── 선택지 토큰 ID (answer-only loss용) ──────────────────────
CHOICE_TOKEN_IDS = set(
    processor.tokenizer.encode(ch, add_special_tokens=False)[0]
    for ch in ["a", "b", "c", "d"]
)



class OptimizedVQADataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.train     = train

        # [최적화 1] 텍스트 프롬프트 재료 미리 준비 (연산 비용 감소)
        self.prompt_materials = []
        for _, row in self.df.iterrows():
            q = str(row["question"])
            opts = (str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"]))
            self.prompt_materials.append((q, opts))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]

        # [최적화 2] 매번 하던 패딩(letterbox) 연산 제거 -> 미리 저장된 이미지 바로 로드! (속도 대폭발)
        # (주의: 첫 번째 위치에 추가했던 precompute_images가 'cached_path' 컬럼을 만들어 둔 상태여야 합니다)
        cached_img_path = row["cached_path"]
        img = Image.open(cached_img_path).convert("RGB")

        # [최적화 3] 동적 프롬프트 증강 (Dynamic Prompt Augmentation)
        q, (a, b, c, d) = self.prompt_materials[i]

        if self.train:
            # 학습 중에는 2가지 버전의 프롬프트를 랜덤하게 섞어서 과적합 방지 (정확도 상승)
            prompt_type = random.choice([1, 2])
            if prompt_type == 1:
                user_text = f"질문: {q}\n(a){a} (b){b} (c){c} (d){d}\n정답을 알파벳 하나로 고르시오."
            else:
                user_text = f"{q}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
        else:
            # 검증/추론 시에는 기존과 동일한 형태 유지
            user_text = f"{q}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."

        # 다른 멤버의 코드 호환성을 위해 원본의 messages 구조 완벽 유지
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text},
            ]},
        ]

        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append(
                {"role": "assistant", "content": [{"type": "text", "text": gold}]}
            )

        return {"messages": messages, "image": img}



@dataclass
class DataCollator:
    """
    [핵심 수정] 프롬프트 구간 마스킹 → Answer 토큰만 loss 반영

    기존: labels = input_ids.clone() → 프롬프트 전체에 loss 계산
    수정: 질문 구간은 -100으로 마스킹 → a/b/c/d 토큰만 loss

    피드백: "라벨에서 프롬프트 구간을 마스킹하고,
             assistant 정답 토큰(한 글자)만 loss 반영"
    """
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"],
                tokenize              = False,
                add_generation_prompt = False,
            )
            texts.append(text)
            images.append(sample["image"])

        enc = self.processor(
            text           = texts,
            images         = images,
            padding        = True,
            return_tensors = "pt",
        )

        if self.train:
            input_ids = enc["input_ids"]           # (B, seq_len)
            labels    = input_ids.clone()

            # ── Answer-Only Loss 마스킹 ─────────────────────
            # a/b/c/d 토큰 위치를 제외한 모든 위치를 -100으로 설정
            # → 모델이 정답 선택지 토큰만 집중적으로 학습
            for i in range(labels.shape[0]):
                answer_pos = None
                for pos in range(labels.shape[1]):
                    if labels[i, pos].item() in CHOICE_TOKEN_IDS:
                        answer_pos = pos
                        break   # 첫 번째 선택지 토큰만 타겟

                if answer_pos is not None:
                    # 선택지 토큰 이전 구간 마스킹
                    labels[i, :answer_pos] = -100
                    # 선택지 토큰 이후 구간도 마스킹
                    # (im_end, \n 등 불필요한 토큰 제외)
                    labels[i, answer_pos + 1:] = -100
                else:
                    # 선택지 토큰 못 찾으면 전체 마스킹 (loss 0)
                    labels[i, :] = -100

            enc["labels"] = labels

        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [5]:
# ============================================================
# Block 6: DataLoader (Pandas DataFrame 슬라이싱 적용)
# ============================================================
import math
from torch.utils.data import DataLoader

# 1. 학습 데이터 줄이기 (선택 사항: 전체를 돌리려면 이 줄을 주석 처리하세요)
# train_subset = train_subset.iloc[:1000].reset_index(drop=True)

# 2. 검증 데이터 줄이기 (필수 권장: 평가 시간 단축을 위해 150개로 제한)
limit_val_len = min(150, len(valid_subset))
valid_subset = valid_subset.iloc[:limit_val_len].reset_index(drop=True)

# 3. 데이터셋 객체 생성 (DataFrame 전달)
train_ds = OptimizedVQADataset(train_subset, processor, train=True)
valid_ds = OptimizedVQADataset(valid_subset, processor, train=True)

# 4. DataLoader 정의
train_loader = DataLoader(
    train_ds,
    batch_size  = BATCH_SIZE, # 추천: 2 (OOM 발생 시 1로 변경)
    shuffle     = True,
    collate_fn  = DataCollator(processor, train=True),
    num_workers = 4,
    pin_memory = True
)
valid_loader = DataLoader(
    valid_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    collate_fn  = DataCollator(processor, train=True),
    num_workers = 2,
)

# 5. 스텝 계산
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps   = int(num_training_steps * WARMUP_RATIO)

print(f"Train batches/epoch : {len(train_loader)}")
print(f"Total training steps: {num_training_steps}")
print(f"Warmup steps        : {num_warmup_steps}")

Train batches/epoch : 846
Total training steps: 846
Warmup steps        : 84


# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [6]:
# ============================================================
# Block 7: Optimizer + 2-Stage Scheduler
# ============================================================

model = model.to(device)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr           = LR,
    weight_decay = WEIGHT_DECAY,
)

# [수정] linear → cosine with warmup
# 피드백: "Stage1: 짧은 warmup + 낮은 LR으로 안정화
#          Stage2: cosine decay + early stopping(accuracy 기준)"
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = num_warmup_steps,
    num_training_steps = num_training_steps,
)

scaler = torch.amp.GradScaler("cuda", enabled=True)

print(f"Optimizer : AdamW (lr={LR}, wd={WEIGHT_DECAY})")
print(f"Scheduler : Cosine with Warmup")
print(f"  warmup  : {num_warmup_steps} steps ({WARMUP_RATIO*100:.0f}%)")
print(f"  total   : {num_training_steps} steps")


Optimizer : AdamW (lr=0.0002, wd=0.01)
Scheduler : Cosine with Warmup
  warmup  : 84 steps (10%)
  total   : 846 steps


In [7]:
# ============================================================
# Block 8: extract_choice (유지 - 우수하다고 판단)
# ============================================================

def extract_choice(text: str) -> str:
    """5단계 폴백 파서 (유지)"""
    t = text.strip()

    # 1차: <ANSWER>x</ANSWER>
    matches = re.findall(r'<ANSWER>([a-d])</ANSWER>', t, re.IGNORECASE)
    if matches:
        return matches[-1].lower()

    t_low = t.lower()

    # 2차: "answer: x" 패턴
    m = re.search(r'answer\s*[:\s=is]+\s*\(?([a-d])\)?', t_low)
    if m:
        return m.group(1)

    # 3차: 마지막 줄 단독 문자
    for line in reversed(t_low.splitlines()):
        line = line.strip()
        if line in ('a', 'b', 'c', 'd'):
            return line

    # 4차: 단어 경계
    m = re.search(r'\b([a-d])\b', t_low)
    if m:
        return m.group(1)

    # 5차: 최초 등장
    for ch in t_low:
        if ch in 'abcd':
            return ch

    return 'a'


In [ ]:
# ============================================================
# Block 9: 학습 루프 (최적화 데이터셋 완벽 호환 버전)
# ============================================================

def evaluate_model(model, processor, valid_subset, valid_loader, device, epoch):
    """val_loss + val_acc 모두 기록 (피드백: 둘 다 기록, best 기준은 acc)"""
    model.eval()

    # Phase 1: Val Loss
    val_loss_total, val_steps = 0.0, 0
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"  [E{epoch}] Val Loss", leave=False):
            vb = {k: v.to(device) for k, v in vb.items()}
            val_loss_total += model(**vb).loss.item()
            val_steps += 1
    val_loss = val_loss_total / val_steps if val_steps > 0 else 0.0

    # Phase 2: Val Accuracy (생성 기반)
    eval_df = valid_subset.reset_index(drop=True)
    correct, total = 0, len(eval_df)

    with torch.no_grad():
        for i in tqdm(range(total), desc=f"  [E{epoch}] Val Acc", leave=False):
            row       = eval_df.iloc[i]

            # [수정 1] 원본 경로 대신 캐싱된 이미지 사용 & 중복 letterbox 연산 제거
            cached_img_path = row["cached_path"]
            img             = Image.open(cached_img_path).convert("RGB")

            # [수정 2] build_mc_prompt 함수 대신 학습 때와 완전히 동일한 프롬프트 사용
            q = str(row["question"])
            a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
            user_text = f"{q}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."

            messages = [
                {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user",   "content": [
                    {"type": "image", "image": img},
                    {"type": "text",  "text": user_text},
                ]},
            ]

            text   = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = processor(
                text=[text], images=[img], return_tensors="pt"
            ).to(device)

            input_len = inputs["input_ids"].shape[1]

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens = MAX_NEW_TOKENS,
                    do_sample      = False,
                    pad_token_id   = processor.tokenizer.eos_token_id,
                )

            new_tokens  = out_ids[0][input_len:]
            output_text = processor.decode(new_tokens, skip_special_tokens=True)
            pred        = extract_choice(output_text)
            true        = str(row["answer"]).strip().lower()

            if pred == true:
                correct += 1

    val_acc = (correct / total * 100) if total > 0 else 0.0
    print(
        f"  📊 [Epoch {epoch}] "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.2f}% ({correct}/{total})"
    )
    model.train()
    return {"val_loss": val_loss, "val_acc": val_acc}


# ── 학습 시작 ─────────────────────────────────────────────────
best_val_acc     = -1.0
best_val_loss    = float("inf")
patience_counter = 0
global_step      = 0
history          = []

print("=" * 50)
print(f"학습 시작: {EPOCHS} Epochs / {num_training_steps} Steps")
print("=" * 50)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running     = 0.0
    epoch_loss  = 0.0
    epoch_steps = 0

    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(
        train_loader,
        desc = f"Epoch {epoch}/{EPOCHS} [train]",
        unit = "batch",
    )

    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss    = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            epoch_loss  += running
            epoch_steps += 1
            running      = 0.0

            pbar.set_postfix({
                "loss": f"{epoch_loss/epoch_steps:.4f}",
                "lr"  : f"{scheduler.get_last_lr()[0]:.2e}",
            })

    avg_train_loss = epoch_loss / epoch_steps if epoch_steps > 0 else 0.0
    print(f"\n[Epoch {epoch}/{EPOCHS}] avg_train_loss = {avg_train_loss:.4f}")

    # ── Validation ───────────────────────────────────────────
    metrics = evaluate_model(
        model, processor, valid_subset, valid_loader, device, epoch
    )

    if metrics["val_acc"] > best_val_acc:
        best_val_acc     = metrics["val_acc"]
        best_val_loss    = metrics["val_loss"]
        patience_counter = 0

        model.save_pretrained(BEST_SAVE_DIR)
        processor.save_pretrained(BEST_SAVE_DIR)

        CONFIG.update({
            "best_val_acc" : best_val_acc,
            "best_val_loss": best_val_loss,
            "best_epoch"   : epoch,
        })
        with open(CKPT_DIR / "config.json", "w") as f:
            json.dump(CONFIG, f, indent=2)

        print(f"  ✅ Best model saved! val_acc={best_val_acc:.2f}% → {BEST_SAVE_DIR}")
    else:
        patience_counter += 1
        print(f"  ⚠️  patience [{patience_counter}/{PATIENCE}]")

    history.append({
        "epoch"     : epoch,
        "train_loss": avg_train_loss,
        "val_loss"  : metrics["val_loss"],
        "val_acc"   : metrics["val_acc"],
    })

    if patience_counter >= PATIENCE:
        print(f"\n🛑 Early Stopping (epoch {epoch})")
        break

    model.train()

# 최종 히스토리 출력
print("\n📊 학습 히스토리")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'Val Acc':>10}")
print("-" * 42)
for h in history:
    mark = " ⭐" if h["val_acc"] == best_val_acc else ""
    print(
        f"{h['epoch']:>6} "
        f"{h['train_loss']:>12.4f} "
        f"{h['val_loss']:>10.4f} "
        f"{h['val_acc']:>9.2f}%"
        f"{mark}"
    )
print(f"\n✅ Best val_acc: {best_val_acc:.2f}%")

학습 시작: 3 Epochs / 846 Steps


Epoch 1/3 [train]:   0%|          | 0/846 [00:00<?, ?batch/s]

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

학습 시작: 1 Epochs / 761 Steps


Epoch 1/1 [train]:   0%|          | 0/761 [00:00<?, ?batch/s]

KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/indexes/base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'cached_path'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^

In [ ]:
# ── Stage 1 학습 종료 및 결과 저장 변수 선언 (기존 코드) ──
print(f" ✅ Stage 1 Best val_acc: {best_val_acc:.2f}%")
best_stage1_acc = best_val_acc
best_incorrect_indices = history[CONFIG["best_epoch"] - 1].get("incorrect_indices", []) if history else []
best_class_prior = history[CONFIG["best_epoch"] - 1].get("class_prior", {'a':1}) if history else {'a':1}

# ============================================================
# Block 9.5: Hard Negative 재학습 (Stage 2)
# ============================================================
if USE_HARD_NEGATIVE_RETRAIN and len(best_incorrect_indices) > 0:
    print("\n" + "=" * 50)
    print("🚀 [기능 1] Stage 2: Hard Negative Retraining 시작")
    print("=" * 50)

    # 1. Hard Set 구성 (검증셋 오답) + Train 믹합
    hard_df = valid_subset.iloc[best_incorrect_indices].copy()
    mix_df = train_subset.sample(frac=HARD_TRAIN_MIX_RATIO, random_state=SEED)
    stage2_train_df = pd.concat([hard_df, mix_df]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    print(f"Hard 샘플: {len(hard_df)}건 | 원본 Train 혼합: {len(mix_df)}건 | 총 2차 학습 데이터: {len(stage2_train_df)}건")

    stage2_train_ds = VQAMCDataset(stage2_train_df, processor, train=True)
    stage2_train_loader = DataLoader(stage2_train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=DataCollator(processor, train=True), num_workers=0)

    # 2. 짧은 스텝 및 낮은 LR 적용
    s2_steps = HARD_EPOCHS * math.ceil(len(stage2_train_loader) / GRAD_ACCUM)
    for param_group in optimizer.param_groups:
        param_group['lr'] = LR * HARD_LR_RATIO

    s2_scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(s2_steps*0.1), num_training_steps=s2_steps)

    best_stage2_acc = -1.0
    s2_save_dir = str(CKPT_DIR / "stage2_best_model")

    # 3. Stage 2 학습 루프
    model.train()
    for step, batch in enumerate(tqdm(stage2_train_loader, desc="Stage 2 Training"), start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss / GRAD_ACCUM
        scaler.scale(loss).backward()

        if step % GRAD_ACCUM == 0 or step == len(stage2_train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            s2_scheduler.step()

    # Stage 2 평가
    print("\nStage 2 Validation 진행 중...")
    s2_metrics = evaluate_model(model, processor, valid_subset, valid_loader, device, epoch="S2")
    best_stage2_acc = s2_metrics["val_acc"]

    print(f"\n✅ Stage 1 Best: {best_stage1_acc:.2f}% VS Stage 2 Best: {best_stage2_acc:.2f}%")
    if best_stage2_acc > best_stage1_acc:
        print("🏆 Stage 2 모델이 더 우수합니다. Stage 2 모델을 최종 저장합니다.")
        model.save_pretrained(BEST_SAVE_DIR)
        processor.save_pretrained(BEST_SAVE_DIR)
        best_class_prior = s2_metrics["class_prior"]
    else:
        print("🛡️ Stage 1 모델 성능이 더 우수하여 기존 Best 모델을 유지/복구합니다.")
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(BEST_SAVE_DIR, device_map=device) # 기존 가중치 롤백

In [ ]:
# ============================================================
# Block 10: 추론 (배치 + 단건 폴백 유지)
# ============================================================

INFER_BATCH = 2
processor.tokenizer.padding_side = 'left'

model.eval()
preds, ids_out = [], []
n = len(test_df)

for start in tqdm(range(0, n, INFER_BATCH), desc="Batch Inference", unit="batch"):
    batch_df = test_df.iloc[start: start + INFER_BATCH]
    batch_texts, batch_images = [], []

    for _, row in batch_df.iterrows():
        try:
            img = Image.open(row["path"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (128, 128, 128))

        # [유지] letterbox 동일 파이프라인 (train/valid/infer 일관)
        img       = letterbox_image(img, target_size=IMAGE_SIZE)
        user_text = build_mc_prompt(
            str(row["question"]),
            str(row["a"]), str(row["b"]),
            str(row["c"]), str(row["d"]),
        )
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text},
            ]},
        ]
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        batch_texts.append(text)
        batch_images.append(img)

    # [유지] 배치 실패 시 단건 폴백
    try:
        inputs = processor(
            text           = batch_texts,
            images         = batch_images,
            padding        = True,
            return_tensors = "pt",
        ).to(device)
    except Exception as e:
        print(f"⚠️ 배치 인코딩 실패 (start={start}): {e} → 단건 처리")
        for t_s, i_s in zip(batch_texts, batch_images):
            inp_s = processor(
                text=[t_s], images=[i_s], return_tensors="pt"
            ).to(device)
            with torch.no_grad():
                out = model.generate(
                    **inp_s,
                    max_new_tokens = MAX_NEW_TOKENS,
                    do_sample      = False,
                )
            dec = processor.decode(
                out[0][inp_s["input_ids"].shape[1]:],
                skip_special_tokens=True,
            )
            preds.append(extract_choice(dec))
        ids_out.extend(batch_df["id"].tolist())
        continue

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            do_sample      = False,
            pad_token_id   = processor.tokenizer.eos_token_id,
        )

    new_tokens   = out_ids[:, input_len:]
    decoded_list = processor.batch_decode(new_tokens, skip_special_tokens=True)

    for decoded in decoded_list:
        preds.append(extract_choice(decoded))
    ids_out.extend(batch_df["id"].tolist())

# ── 제출 파일 ─────────────────────────────────────────────────
submission = pd.DataFrame({"id": ids_out, "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print(f"Saved /content/submission.csv ({len(submission)}건)")
print("정답 분포:\n", submission["answer"].value_counts().sort_index())
